In [1]:
import torch
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
iris = load_iris()
X, y = iris.data, iris.target

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [3]:
class Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = torch.randn(n_inputs, n_neurons) * 0.1
        self.biases = torch.zeros(n_neurons)

    def forward(self, inputs):
        self.inputs = inputs
        self.output = inputs @ self.weights + self.biases
        return self.output

    def backward(self, dvalues, lr=0.01):
        self.dweights = self.inputs.T @ dvalues
        self.dbiases = dvalues.sum(axis=0)
        self.dinputs = dvalues @ self.weights.T
        self.weights -= lr * self.dweights
        self.biases -= lr * self.dbiases

class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = inputs.clamp(min=0)
        return self.output

    def backward(self, dvalues):
        self.dinputs = dvalues.clone()
        self.dinputs[self.inputs <= 0] = 0

class Activation_Softmax:
    def forward(self, inputs):
        exp = torch.exp(inputs - torch.max(inputs, dim=1, keepdim=True).values)
        self.output = exp / torch.sum(exp, dim=1, keepdim=True)
        return self.output

    def backward(self, dvalues):
        self.dinputs = dvalues

class Loss_CategoricalCrossEntropy:
    def forward(self, y_pred, y_true):
        samples = len(y_pred)
        y_pred_clipped = torch.clip(y_pred, 1e-7, 1-1e-7)
        correct_confidences = y_pred_clipped[range(samples), y_true]
        neg_log_likelihoods = -torch.log(correct_confidences)
        return torch.mean(neg_log_likelihoods)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        self.dinputs = dvalues.clone()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs / samples

In [4]:
fc1 = Dense(4, 10)
act1 = Activation_ReLU()
fc2 = Dense(10, 3)
act2 = Activation_Softmax()
loss_fn = Loss_CategoricalCrossEntropy()

In [5]:
epochs = 2000
lr = 0.01

for epoch in range(epochs):
    # forward
    out = fc1.forward(X_train)
    out = act1.forward(out)
    out = fc2.forward(out)
    out = act2.forward(out)

    # loss
    loss = loss_fn.forward(out, y_train)

    # accuracy
    preds = torch.argmax(out, dim=1)
    acc = (preds == y_train).float().mean()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, loss={loss.item():.4f}, acc={acc.item():.4f}")

    # backward
    loss_fn.backward(out, y_train)
    fc2.backward(loss_fn.dinputs, lr)
    act1.backward(fc2.dinputs)
    fc1.backward(act1.dinputs, lr)

Epoch 0, loss=1.1046, acc=0.3333
Epoch 20, loss=1.0907, acc=0.3333
Epoch 40, loss=1.0780, acc=0.4583
Epoch 60, loss=1.0665, acc=0.6667
Epoch 80, loss=1.0567, acc=0.6750
Epoch 100, loss=1.0447, acc=0.7083
Epoch 120, loss=1.0300, acc=0.7167
Epoch 140, loss=1.0128, acc=0.7167
Epoch 160, loss=0.9939, acc=0.7167
Epoch 180, loss=0.9731, acc=0.7083
Epoch 200, loss=0.9504, acc=0.7083
Epoch 220, loss=0.9257, acc=0.7167
Epoch 240, loss=0.8989, acc=0.7333
Epoch 260, loss=0.8701, acc=0.7417
Epoch 280, loss=0.8395, acc=0.7500
Epoch 300, loss=0.8075, acc=0.7583
Epoch 320, loss=0.7744, acc=0.7833
Epoch 340, loss=0.7410, acc=0.7833
Epoch 360, loss=0.7080, acc=0.8000
Epoch 380, loss=0.6762, acc=0.8000
Epoch 400, loss=0.6462, acc=0.8083
Epoch 420, loss=0.6183, acc=0.8083
Epoch 440, loss=0.5929, acc=0.8083
Epoch 460, loss=0.5698, acc=0.8083
Epoch 480, loss=0.5490, acc=0.8167
Epoch 500, loss=0.5301, acc=0.8167
Epoch 520, loss=0.5130, acc=0.8167
Epoch 540, loss=0.4974, acc=0.8167
Epoch 560, loss=0.4831, ac

In [6]:
out = fc1.forward(X_test)
out = act1.forward(out)
out = fc2.forward(out)
out = act2.forward(out)
preds = torch.argmax(out, dim=1)
test_acc = (preds == y_test).float().mean()
print(f"Test accuracy = {test_acc.item():.4f}")

Test accuracy = 0.9333
